# Meyaar — GeoSA RAG

هذا النوتبوك يبني **GeoSA Retrieval-Augmented Generation (RAG)** لمشروع Meyaar.

**Validation Result → Retrieve GeoSA Evidence → LLM Explanation → Citation + Recommendation**

> الـRAG هنا لا يكتشف الخطأ. الـRule Engine أو المودل يكتشف الحالة أولًا، ثم الـRAG يبحث عن النص المناسب من مستندات GeoSA ويشرح النتيجة مع المصدر.

### المخرجات
- قراءة PDF / DOCX
- Chunking
- Multilingual embeddings
- FAISS retrieval
- Grounded Arabic answer
- Source/page citations
- Integration function لنتائج Meyaar
- RAG evaluation


## 1. Install dependencies

In [ ]:
!pip -q install pymupdf python-docx sentence-transformers faiss-cpu transformers accelerate pandas numpy tqdm

## 2. Imports and reproducibility

In [ ]:
import os, re, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import fitz
from docx import Document
from tqdm.auto import tqdm
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 3. GeoSA document paths

ارفعي مستندات GeoSA في Colab داخل `/content/`.

المقترح:
- National Geospatial Data Standards
- Technical Guidelines for Cartography
- SANSRS User Guideline
- KSA National Feature Concept Dictionary
- KSA National Geospatial Metadata Profile
- National Geospatial Data Governance Framework


In [ ]:
CONTENT_DIR = Path("/content")

GEOSA_DOCS = sorted(
    list(CONTENT_DIR.glob("*.pdf")) +
    list(CONTENT_DIR.glob("*.docx"))
)

print(f"Found {len(GEOSA_DOCS)} document(s):")
for p in GEOSA_DOCS:
    print(" -", p.name)

assert GEOSA_DOCS, "Upload GeoSA PDF/DOCX files into /content first."


## 4. Extract text with source metadata

In [ ]:
def clean_text(text):
    text = text.replace("\u00ad", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_pdf(path):
    rows = []
    doc = fitz.open(path)
    for page_num, page in enumerate(doc, start=1):
        text = clean_text(page.get_text("text"))
        if len(text) >= 20:
            rows.append({
                "source": path.name,
                "page": page_num,
                "section": None,
                "text": text
            })
    return rows

def extract_docx(path):
    doc = Document(path)
    rows, buffer = [], []
    current_heading = None

    def flush():
        nonlocal buffer
        text = clean_text(" ".join(buffer))
        if len(text) >= 20:
            rows.append({
                "source": path.name,
                "page": None,
                "section": current_heading,
                "text": text
            })
        buffer = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        style = para.style.name.lower() if para.style else ""
        if "heading" in style:
            flush()
            current_heading = text
        else:
            buffer.append(text)

    flush()
    return rows

def extract_document(path):
    if path.suffix.lower() == ".pdf":
        return extract_pdf(path)
    if path.suffix.lower() == ".docx":
        return extract_docx(path)
    return []

raw_records = []
for path in tqdm(GEOSA_DOCS):
    try:
        raw_records.extend(extract_document(path))
    except Exception as e:
        print(f"Could not read {path.name}: {e}")

raw_df = pd.DataFrame(raw_records)
print("Extracted units:", len(raw_df))
display(raw_df.head())


## 5. Chunking

القيم هنا إعدادات RAG تجريبية وليست متطلبات GeoSA.


In [ ]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]

        if end < len(text):
            cut = max(
                chunk.rfind("."),
                chunk.rfind("؟"),
                chunk.rfind("!")
            )
            if cut > int(chunk_size * 0.55):
                end = start + cut + 1
                chunk = text[start:end]

        chunk = clean_text(chunk)
        if len(chunk) >= 50:
            chunks.append(chunk)

        if end >= len(text):
            break
        start = max(end - overlap, start + 1)

    return chunks

chunk_records = []
for _, row in raw_df.iterrows():
    for chunk_id, chunk in enumerate(split_text(row["text"])):
        chunk_records.append({
            "source": row["source"],
            "page": row["page"],
            "section": row["section"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunk_records)
print("Total chunks:", len(chunks_df))
display(chunks_df.head())


## 6. Multilingual embeddings

`intfloat/multilingual-e5-small` يدعم retrieval بالعربي والإنجليزي.


In [ ]:
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

passages = ["passage: " + t for t in chunks_df["text"].tolist()]

embeddings = embedding_model.encode(
    passages,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")
print("Embeddings shape:", embeddings.shape)


## 7. Build FAISS index

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS vectors:", index.ntotal)


## 8. Retrieval function

In [ ]:
def retrieve(query, top_k=5):
    q = embedding_model.encode(
        ["query: " + query],
        normalize_embeddings=True
    )
    q = np.asarray(q, dtype="float32")
    scores, indices = index.search(q, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[int(idx)]
        results.append({
            "score": float(score),
            "source": row["source"],
            "page": row["page"],
            "section": row["section"],
            "chunk_id": int(row["chunk_id"]),
            "text": row["text"]
        })
    return results

def show_retrieval(results):
    for i, r in enumerate(results, 1):
        print("=" * 90)
        print(f"[{i}] score={r['score']:.4f}")
        print("Source:", r["source"])
        if pd.notna(r["page"]):
            print("Page:", int(r["page"]))
        if r["section"]:
            print("Section:", r["section"])
        print()
        print(r["text"][:1500])
        print()


## 9. Test retrieval first

In [ ]:
query = "ما المتطلبات المتعلقة بنظام الإسناد المكاني أو المرجع الجيوديسي في السعودية؟"
results = retrieve(query, top_k=5)
show_retrieval(results)


## 10. Load a small instruction LLM

نستخدم Qwen 1.5B للـPoC. على CPU سيكون أبطأ.


In [ ]:
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

if torch.cuda.is_available():
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        torch_dtype=torch.float32
    ).to("cpu")

print("Loaded:", LLM_NAME)


## 11. Build context with source IDs

In [ ]:
def make_context(results):
    blocks = []
    for i, r in enumerate(results, 1):
        page = f"page {int(r['page'])}" if pd.notna(r["page"]) else "page unavailable"
        section = f", section: {r['section']}" if r["section"] else ""
        blocks.append(
            f"[S{i}] Source: {r['source']}, {page}{section}\n{r['text']}"
        )
    return "\n\n".join(blocks)


## 12. Grounded GeoSA answer function

In [ ]:
SYSTEM_PROMPT = """
أنت مساعد امتثال جيومكاني داخل نظام Meyaar.

قواعد إلزامية:
- استخدم فقط المعلومات الموجودة في المصادر المسترجعة.
- لا تخترع معيارًا أو رقمًا أو threshold.
- لا تقل إن شيئًا مخالف لـ GeoSA إلا إذا كان النص المسترجع يدعم ذلك.
- ضع citation بصيغة [S1] أو [S2] بعد الادعاء الذي تدعمه.
- إذا لم تجد أدلة كافية، قل:
  "لم أجد في المقاطع المسترجعة دليلًا كافيًا للحكم على هذه الحالة."
- فرّق بين نص المصدر وبين تفسير Meyaar أو التوصية.
- أجب بالعربية بشكل مختصر وواضح.
"""

def generate_answer(question, top_k=5, max_new_tokens=350):
    retrieved = retrieve(question, top_k=top_k)
    context = make_context(retrieved)

    user_prompt = f"""
السؤال:
{question}

المصادر المسترجعة:
{context}

أجب بالصيغة:
الحكم/الخلاصة:
...

الدليل:
...

التفسير:
...

التوصية:
...

المصادر المستخدمة:
[S?], [S?]
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer.strip(),
        "retrieved": retrieved
    }


## 13. Ask GeoSA RAG

In [ ]:
response = generate_answer(
    "ما الذي تقوله المستندات عن جودة البيانات الجيومكانية والالتزام بالمعايير؟"
)
print(response["answer"])


## 14. Inspect retrieved evidence

In [ ]:
show_retrieval(response["retrieved"])

# 15. Connect RAG to Meyaar validation results

مثال input من المودل أو Rule Engine:


In [ ]:
def explain_validation_result(result, top_k=5):
    feature_id = result.get("feature_id", "unknown")
    dataset_type = result.get("dataset_type", "unknown")
    error_type = result.get("error_type", "unknown")
    detector = result.get("detector", "unknown")
    confidence = result.get("confidence")

    confidence_text = (
        f"{confidence:.1%}"
        if isinstance(confidence, (float, int))
        else "not provided"
    )

    question = f"""
تم اكتشاف حالة محتملة في بيانات جيومكانية.

Feature ID: {feature_id}
Dataset type: {dataset_type}
Detected issue: {error_type}
Detector: {detector}
Model confidence: {confidence_text}

ابحث في مستندات GeoSA عن المتطلبات أو التوجيهات ذات الصلة بهذه الحالة.
لا تفترض أن اسم الخطأ نفسه موجود حرفيًا في المعيار.
اشرح فقط العلاقة التي تدعمها المصادر المسترجعة.
إذا لم يوجد نص كافٍ لإثبات مخالفة رسمية، اذكر ذلك بوضوح.
"""
    return generate_answer(question, top_k=top_k)


## 16. Example — Road Undershoot

In [ ]:
example_result = {
    "feature_id": "road_demo_001",
    "dataset_type": "roads",
    "error_type": "undershoot",
    "detector": "XGBoost",
    "confidence": 0.91
}

rag_result = explain_validation_result(example_result)
print(rag_result["answer"])


## 17. Example — Building Overlap

In [ ]:
building_result = {
    "feature_id": "building_demo_001",
    "dataset_type": "buildings",
    "error_type": "overlap",
    "detector": "Geo2Vec + MLP",
    "confidence": None
}

rag_building = explain_validation_result(building_result)
print(rag_building["answer"])


## 18. Save FAISS index + chunks + config

In [ ]:
OUTPUT_DIR = Path("/content/meyaar_geosa_rag")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

faiss.write_index(index, str(OUTPUT_DIR / "geosa_faiss.index"))

chunks_df.to_json(
    OUTPUT_DIR / "geosa_chunks.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "llm_model": LLM_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP
}

with open(OUTPUT_DIR / "rag_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved to:", OUTPUT_DIR)


# 19. RAG Evaluation

المقاييس:
- Retrieval Hit@K
- Citation Correctness
- Hallucination Rate

ابدئي بـ20–30 سؤالًا يدويًا من المستندات.


In [ ]:
evaluation_questions = pd.DataFrame([
    {
        "question": "ما المرجع أو التوجيه المتعلق بنظام الإسناد المكاني في السعودية؟",
        "expected_source_keyword": "SANSRS"
    },
    {
        "question": "ما المستند المرتبط بإرشادات cartography؟",
        "expected_source_keyword": "Cartography"
    },
    {
        "question": "ما المصدر المرتبط بمفاهيم المعالم الجيومكانية؟",
        "expected_source_keyword": "Feature"
    }
])

evaluation_questions


In [ ]:
def retrieval_hit_at_k(question, expected_keyword, top_k=5):
    results = retrieve(question, top_k=top_k)
    haystack = " ".join(
        str(r["source"]) + " " +
        str(r.get("section", "")) + " " +
        str(r["text"])
        for r in results
    ).lower()
    return expected_keyword.lower() in haystack

eval_rows = []
for _, row in evaluation_questions.iterrows():
    eval_rows.append({
        "question": row["question"],
        "expected_source_keyword": row["expected_source_keyword"],
        "hit@5": retrieval_hit_at_k(
            row["question"],
            row["expected_source_keyword"],
            top_k=5
        )
    })

retrieval_eval_df = pd.DataFrame(eval_rows)
display(retrieval_eval_df)
print("Retrieval Hit@5:", retrieval_eval_df["hit@5"].mean())


## 20. Manual citation / hallucination evaluation template

In [ ]:
manual_eval_template = pd.DataFrame(columns=[
    "question",
    "answer",
    "retrieved_sources",
    "retrieval_correct",
    "citation_correct",
    "hallucination_present",
    "notes"
])

manual_eval_template


# 21. Final Meyaar RAG flow

```text
Rule Engine / ML Detector
          ↓
   Detected Finding
          ↓
Build retrieval query
          ↓
Multilingual E5
          ↓
       FAISS
          ↓
Top-K GeoSA Evidence
          ↓
Qwen Instruct
          ↓
Issue → Evidence → Standard → Explanation → Recommendation
```

**صياغة العرض:**

> The validator detects the issue first. GeoSA RAG then retrieves relevant standards evidence and produces a source-grounded explanation and recommendation.
